# 08 — Multi-Task Models

## Goals
Train models that simultaneously predict multiple injury-related outcomes, leveraging
shared information across tasks. Multi-task learning is particularly useful here
because injury probability, injury type, and severity are causally related.

## Inputs
- `data/processed/feature_matrix.parquet` (from notebook 05)

## Outputs
- `models/multitask_chained.joblib`
- `models/multitask_shared.joblib`
- `reports/tables/multitask_model_results.csv`

## Modeling Targets (Tasks)
1. `injured_within_30d` — binary classification
2. `injured_within_60d` — binary classification
3. `injured_within_90d` — binary classification
4. `next_injury_type` — multiclass classification (elbow, shoulder, forearm, other)
5. `next_days_lost` — regression
6. `days_until_next_injury` — regression / survival

## Planned Analyses

### Architecture Comparison
1. **Chained multi-task model:** Train injury probability first, then pass predictions
   as features into injury-type and severity models. Reflects causal structure.
2. **Shared representation:** Single multi-output model (sklearn MultiOutputClassifier
   or multi-output XGBoost) with shared tree structure.

### Evaluation
1. Per-task metrics: AUC-ROC, PR-AUC, Brier (classification); MAE, RMSE, R² (regression)
2. Compare against single-task baselines from notebook 06
3. Analyze which tasks benefit most from multi-task learning
4. Temporal cross-validation for stability

## Future Work
- Implement neural multi-task model (shared trunk with task-specific heads) in PyTorch
- Experiment with task weighting in the loss function
- Evaluate uncertainty quantification (prediction intervals per task)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.models.multitask_models import (
    prepare_multitask_dataset,
    train_chained_multitask_model,
    train_shared_representation_model,
    predict_all_tasks,
    compute_multitask_metrics,
)

# TODO: load feature matrix and run multi-task model pipeline